In [ ]:
# run first three cells then find the metrics cell and run than before running anything at the bottom

In [1]:
label2id = {"left": 0, "center": 1, "right": 2}
id2label = {v: k for k, v in label2id.items()}

In [10]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128#512
    )


In [3]:
# load data from our file
from data_loader import load_split

#data_dir = "/Users/sarah/Documents/SMU/NLP/Article-Bias-Prediction-main/data"
data_dir = "C:\\Users\\nia_4\\SMU\\8sem\\nlp\\project\\data"

X_train, y_train = load_split(data_dir, "media", "train")
X_val, y_val = load_split(data_dir, "media", "valid")
X_test, y_test = load_split(data_dir, "media", "test")

print(len(X_train), len(X_val), len(X_test))

26590 2356 1300


In [4]:
import pandas as pd

train_df = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

val_df = pd.DataFrame({
    "text": X_val,
    "label": y_val
})

test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test
})


In [5]:
print(y_train[:5])

[2, 1, 0, 1, 0]


In [6]:
print(train_df["label"].value_counts())
print(val_df["label"].value_counts())
print(test_df["label"].value_counts())


label
2    10241
0     8861
1     7488
Name: count, dtype: int64
label
0    1640
1     618
2      98
Name: count, dtype: int64
label
2    599
0    402
1    299
Name: count, dtype: int64


In [7]:
train_df.head()

,text,label
0,President Trump and Senate Minority Leader Chu...,2
1,“ The 360 ” shows you diverse perspectives on ...,1
2,LOS ANGELES — Actress Rosario Dawson took the ...,0
3,President Donald Trump said on Friday that he ...,1
4,Washington ( CNN ) Donald Trump became the 45t...,0


In [8]:
# sample datasets down to get faster training & eval
def quick_balanced_sample(df, n_per_class=500, random_state=42):
    samples = []
    
    for label in df["label"].unique():
        subset = df[df["label"] == label]
        
        sampled_subset = subset.sample(
            n=min(len(subset), n_per_class),
            random_state=random_state
        )
        
        samples.append(sampled_subset)
    
    # Combine and shuffle
    balanced_df = (
        pd.concat(samples)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )
    
    return balanced_df

In [9]:
train_sampled = quick_balanced_sample(train_df, 500)
val_sampled   = quick_balanced_sample(val_df, 200)
test_sampled  = quick_balanced_sample(test_df, 200)


In [10]:
print(train_sampled["label"].value_counts())
print(val_sampled["label"].value_counts())
print(test_sampled["label"].value_counts())


label
0    500
2    500
1    500
Name: count, dtype: int64
label
1    200
0    200
2     98
Name: count, dtype: int64
label
2    200
0    200
1    200
Name: count, dtype: int64


In [11]:
train_sampled.head()

,text,label
0,"Julian Assange insists , against all evidence ...",0
1,This is a rush transcript . Copy may not be in...,0
2,Iran on Saturday reportedly released Washingto...,2
3,Not even six months into its existence ObamaCa...,2
4,Is the Obamacare coalition finally cracking ? ...,2


In [12]:
from datasets import Dataset

# train_dataset = Dataset.from_pandas(train_df)
# val_dataset = Dataset.from_pandas(val_df)
# test_dataset = Dataset.from_pandas(test_df)

train_dataset = Dataset.from_pandas(train_sampled)
val_dataset = Dataset.from_pandas(val_sampled)
test_dataset = Dataset.from_pandas(test_sampled)



In [13]:
from datasets import DatasetDict

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

In [14]:
print(dataset.column_names)

{'train': ['text', 'label'], 'validation': ['text', 'label'], 'test': ['text', 'label']}


In [15]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

dataset = dataset.map(tokenize, batched=True)

Map: 100%|██████████| 600/600 [00:00<00:00, 1585.07 examples/s]


In [16]:
for split in dataset:
    dataset[split].set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "label"]
    )

In [17]:
import torch
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [18]:
from transformers import RobertaForSequenceClassification

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1192.89it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"  # or "macro"
    )
    
    acc = accuracy_score(labels, predictions)
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [20]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    #evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    #tokenizer=tokenizer
    compute_metrics=compute_metrics
)

#trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    #tokenizer=tokenizer
)

In [ ]:
#will prolly take 3 hrs+ to run so dont run this cell if you dont have time
trainer.train()

c:\Users\nia_4\OneDrive\Documents\GitHub\NLP_Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.47s/it]
c:\Users\nia_4\OneDrive\Documents\GitHub\NLP_Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [23]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

def classify(text):
    model.eval()
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )
    
    # 🔑 Move inputs to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits = outputs.logits
    predicted_class_id = torch.argmax(logits, dim=1).item()
    
    return id2label[predicted_class_id]

In [29]:
#print(len(X_val_feat))
print(len(dataset["validation"]["text"]))

498


In [24]:
print(classify("The policy is bad because it's extrememly bad for the environent. This is a regressive policy that will lead to more pollution and harm our planet."))
# → "right" / "left" / "center"

right


In [ ]:
predictions = trainer.predict(dataset["test"])

import numpy as np
from sklearn.metrics import classification_report

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print(classification_report(y_true, y_pred, target_names=["left", "center", "right"]))
#4min run


              precision    recall  f1-score   support

        left       0.00      0.00      0.00       200
      center       0.12      0.01      0.03       200
       right       0.34      0.98      0.51       200

    accuracy                           0.33       600
   macro avg       0.15      0.33      0.18       600
weighted avg       0.15      0.33      0.18       600



c:\Users\nia_4\OneDrive\Documents\GitHub\NLP_Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nia_4\OneDrive\Documents\GitHub\NLP_Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nia_4\OneDrive\Documents\GitHub\NLP_Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _

In [ ]:
from scipy.special import softmax
val_preds = trainer.predict(dataset["validation"])
val_logits = val_preds.predictions
val_probs = softmax(val_logits, axis=1)

# test preds
test_preds = trainer.predict(dataset["test"])
test_logits = test_preds.predictions
test_probs = softmax(test_logits, axis=1)

y_true = test_preds.label_ids
y_pred = np.argmax(test_logits, axis=1)

print(classification_report(y_true, y_pred, target_names=["left", "center", "right"]))

np.save("roberta_val_probs.npy", val_probs)
np.save("roberta_test_probs.npy", test_probs)
#~7min run

              precision    recall  f1-score   support

        left       0.00      0.00      0.00       200
      center       0.12      0.01      0.03       200
       right       0.34      0.98      0.51       200

    accuracy                           0.33       600
   macro avg       0.15      0.33      0.18       600
weighted avg       0.15      0.33      0.18       600



c:\Users\nia_4\OneDrive\Documents\GitHub\NLP_Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nia_4\OneDrive\Documents\GitHub\NLP_Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nia_4\OneDrive\Documents\GitHub\NLP_Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _

In [10]:
import numpy as np
from scipy.special import softmax

# val preds
val_preds = trainer.predict(dataset["validation"])
val_logits = val_preds.predictions
val_probs = softmax(val_logits, axis=1)

# test preds
test_preds = trainer.predict(dataset["test"])
test_logits = test_preds.predictions
test_probs = softmax(test_logits, axis=1)

np.save("roberta_val_probs.npy", val_probs)
np.save("roberta_test_probs.npy", test_probs)
np.save("roberta_val_labels.npy", val_preds.label_ids)
#

NameError: name 'trainer' is not defined

In [84]:
dataset["test"].column_names


['text', 'label', 'input_ids', 'attention_mask']

## redoing to make sure poc ensemble works

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

X_train, y_train = load_split(data_dir, "media", "train")
X_val, y_val = load_split(data_dir, "media", "valid")
X_test, y_test = load_split(data_dir, "media", "test")

from datasets import Dataset

train_dataset = Dataset.from_dict({
    "text": X_train,
    "label": y_train
})

val_dataset = Dataset.from_dict({
    "text": X_val,
    "label": y_val
})

test_dataset = Dataset.from_dict({
    "text": X_test,
    "label": y_test
})


In [11]:
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 1300/1300 [00:01<00:00, 819.29 examples/s] 


In [12]:
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [13]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cpu
False


In [20]:
import numpy as np
from scipy.special import softmax
from sklearn.metrics import classification_report
from transformers import logging
from transformers import TrainingArguments, Trainer

from transformers import RobertaForSequenceClassification

model = RobertaForSequenceClassification.from_pretrained(
    #"roberta-base",
    "distilroberta-base",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=1,#3,
    #evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    #use a random sample of the data for faster training
    train_dataset=train_dataset.shuffle(seed=42).select(range(int(len(train_dataset)/4))),   
    eval_dataset=val_dataset,      
    compute_metrics=compute_metrics
)



loading configuration file config.json from cache at C:\Users\nia_4\.cache\huggingface\hub\models--distilroberta-base\snapshots\fb53ab8802853c8e4fbdbcd0529f21fc6f459b2b\config.json
Model config RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "left",
    "1": "center",
    "2": "right"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "center": 1,
    "left": 0,
    "right": 2
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.5.4",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265


In [21]:
logging.set_verbosity_info()

trainer.train()
#~25min run

The following columns in the Training set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: text. If text are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 6,647
  Num Epochs = 1
  Num update steps per epoch = 208
  Instantaneous batch size per device = 32
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 1
  Total optimization steps = 208
  Number of trainable parameters = 82,120,707
c:\Users\nia_4\OneDrive\Documents\GitHub\NLP_Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Saving model checkpoint to ./results\checkpoint-208
Configuration saved in ./results\checkpoint-208\config.json
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.49it/s]
Model weights saved in ./results\checkpoint-208\model.safetensors


Training completed. Do not forget to share your model on huggingface.co/models =)




TrainOutput(global_step=208, training_loss=0.9036511641282302, metrics={'train_runtime': 1389.2653, 'train_samples_per_second': 4.785, 'train_steps_per_second': 0.15, 'total_flos': 220131625381632.0, 'train_loss': 0.9036511641282302, 'epoch': 1.0})

In [22]:
logging.set_verbosity_info()
#add progress bar
trainer.progress_bar = True
# val probs
val_outputs = trainer.predict(val_dataset)
val_logits = val_outputs.predictions
val_probs = softmax(val_logits, axis=1)
#15-20min run

The following columns in the test set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: text. If text are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 2356
  Batch size = 32
c:\Users\nia_4\OneDrive\Documents\GitHub\NLP_Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [23]:
logging.set_verbosity_info()
# test probs
test_outputs = trainer.predict(test_dataset)
test_logits = test_outputs.predictions
test_probs = softmax(test_logits, axis=1)

The following columns in the test set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: text. If text are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 1300
  Batch size = 32
c:\Users\nia_4\OneDrive\Documents\GitHub\NLP_Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [24]:

y_true = test_outputs.label_ids
y_pred = np.argmax(test_logits, axis=1)

print(classification_report(y_true, y_pred))

np.save("roberta_val_probs.npy", val_probs)
np.save("roberta_test_probs.npy", test_probs)

              precision    recall  f1-score   support

           0       0.63      0.51      0.56       402
           1       0.29      0.23      0.25       299
           2       0.60      0.74      0.66       599

    accuracy                           0.55      1300
   macro avg       0.51      0.49      0.49      1300
weighted avg       0.54      0.55      0.54      1300

